# Notebook 04 — Modules, JSON & pathlib

| | |
|---|---|
| **Track** | AI Engineering · `03-python-foundations/exercises/` |
| **Level** | Intermediate |
| **Pattern** | *Concept* → *Runnable demo* → *Your turn* → *Solution* |

**Prerequisites:** `03-control-flow-functions.ipynb`

**Next up:** `05-classes-oop.ipynb`

---

## Learning objectives

- Import cleanly and guard `main` execution.
- Parse and emit JSON safely.
- Use `pathlib.Path` for portable IO — RAG ingestion starts here.

## Table of contents

1. **Imports & __main__**
2. **JSON payloads**
3. **Path & text files**
4. **Progressive drills — JSON types → extensions → pretty dumps**
5. **Exercise — JSONL count**

---

## How to use this notebook

1. Run cells **top to bottom** the first time through.
2. Re-run and **change values** to test your understanding.
3. Do **Your turn** cells before opening solutions (HTML `<details>` blocks or later cells).

---


## 1 · Import discipline

*Explanation:* `if __name__ == "__main__":` keeps modules importable from tests.


In [ ]:
import json
from pathlib import Path

print(Path.cwd())


## 2 · JSON for tool calls

*Explanation:* Models return JSON-shaped tool calls — practice `loads`/`dumps`.


In [ ]:
payload = {"tool": "search", "args": {"q": "vector db"}}
raw = json.dumps(payload, ensure_ascii=False)
print(raw, "->", json.loads(raw)["tool"])


## 3 · Path read/write

*Explanation:* Always pass `encoding="utf-8"` explicitly for text.


In [ ]:
p = Path("_nb_tmp_demo.txt")
p.write_text("hello\nworld", encoding="utf-8")
print(p.read_text(encoding="utf-8").splitlines())
p.unlink(missing_ok=True)


---

## Progressive drills — **easy → harder**

Files + JSON underpin **every ingestion pipeline**. Ramp from typed parsing → filesystem hygiene → readable logs.


### A · Easiest — detect JSON container types

Tool outputs might be arrays **or** objects—branch safely.


In [ ]:
import json

samples = ["[]", '{"hits":["a"]}', '"plain"']
for raw in samples:
    obj = json.loads(raw)
    kind = "array" if isinstance(obj, list) else "object" if isinstance(obj, dict) else "scalar"
    print(raw, "->", kind)


### B · Medium — filter ingestion candidates by suffix

Skip `.png`, ingest `.jsonl` only.


In [ ]:
from pathlib import Path

paths = [Path("notes.txt"), Path("dataset.jsonl"), Path("manifest.JSON")]
targets = [p for p in paths if p.suffix.lower() == ".jsonl"]
print([p.name for p in targets])


### C · Harder — stable JSON logs (`sort_keys`, `indent`)

Diff-friendly dumps help observability.


In [ ]:
import json

payload = {"model": "mini", "params": {"temperature": 0.2, "top_k": 8}}
print(json.dumps(payload, sort_keys=True, indent=2))


### Exercise — JSONL count

Implement `count_objects(path: Path) -> int`: number of lines that are non-empty **and** valid JSON **objects** (`dict`).


In [ ]:
import json
from pathlib import Path


def count_objects(path: Path) -> int:
    raise NotImplementedError


tmp = Path("_nb_jsonl_demo.jsonl")
tmp.write_text('{"k":1}

[1,2]
{"ok":true}
', encoding="utf-8")
assert count_objects(tmp) == 2
tmp.unlink(missing_ok=True)
print("OK")


### Solution — JSONL count

<details>
<summary>Click to expand</summary>

```python
def count_objects(path: Path) -> int:
    n = 0
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        try:
            val = json.loads(line)
        except json.JSONDecodeError:
            continue
        if isinstance(val, dict):
            n += 1
    return n
```

</details>
